# 06 · M2: regresión logística jerárquica (MH por componentes + Gibbs)
**Grupo 2 · Estadística Bayesiana · Entrega final**  
Jean Paul Rojas Rojas · Daniel Estiven Pacheco Villagrande · Diego Andrés Cascante Camacho

**Qué hace.** Define y corre el muestreador de M2 con efectos de localidad, franja, interacción y año; calcula R̂ y ESS y guarda la severidad predicha para un año nuevo, que es la base de la validación.

**Entradas.** `datos/siniestros_limpios.csv` (celdas localidad × franja × año 2015–2017).

**Salidas.** `salidas/resultados/diagnosticos_M2.csv`, `06_M2.json`, `m2_resumen.npz`.

*Respaldo por etapas: el notebook principal es `../Grupo2_Bayesiana_completo.ipynb`, que reúne estas ocho etapas.* Orden de ejecución y trazabilidad completa: `../README.md`.

In [1]:
# Configuración: funciona en Google Colab y en un computador
# En Colab: suba la carpeta "notebooks" a su Drive. Si el buscador automático no la encuentra,
# escriba su ruta en BASE_COLAB, p. ej. '/content/drive/MyDrive/Grupo2_EntregaFinal_Bayesiana/notebooks'
BASE_COLAB = ''
import os, sys, glob
from pathlib import Path
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not BASE_COLAB:
        cand = glob.glob('/content/drive/MyDrive/**/notebooks/comun.py', recursive=True)
        assert cand, 'No encontré la carpeta notebooks en su Drive: escriba su ruta en BASE_COLAB'
        BASE_COLAB = os.path.dirname(cand[0])
    BASE = Path(BASE_COLAB)
else:
    # el notebook puede estar en notebooks/ o en notebooks/por_etapas/
    BASE = Path.cwd() if (Path.cwd() / 'comun.py').exists() else Path.cwd().parent
os.chdir(BASE)
sys.path.insert(0, str(BASE))
import numpy, pandas, scipy, matplotlib
print('Carpeta:', BASE)
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)

Carpeta: C:\Users\dddan\Downloads\Pensamiento - Bayesiana\4. Estadística Bayesiana\Proyecto\4. Entrega final\notebooks
Python 3.12.10 | numpy 2.3.5 | pandas 2.3.3 | scipy 1.18.1 | matplotlib 3.10.7


In [2]:
from comun import *
import time
import arviz as az
from mcmc import adaptar, resumen_az
print('arviz', az.__version__)

arviz 0.23.4


## El algoritmo
Metropolis-Hastings por componentes (vectorizado), movimientos de traslación entre niveles y Gibbs (Inversa-Gamma) para las varianzas. Idéntico a `mcmc.py` y al script final.

In [3]:
# ---------------------------------------------------------------------------
# 5. M2: logística jerárquica
#   y_{lft} | theta ~ Bin(n_{lft}, theta_{lft}),  logit theta = mu + a_l + b_f + g_lf + d_t
#   mu ~ N(logit(1/3), 1);  a_l ~ N(0, s2a);  b_f ~ N(0, s2b);  g_lf ~ N(0, s2g);  d_t ~ N(0, s2d)
#   s2a ~ IG(3, 0.5), s2b ~ IG(3, 0.5), s2g ~ IG(3, 0.1), s2d ~ IG(3, 0.05)
# ---------------------------------------------------------------------------
M0, S0 = sp.logit(1 / 3), 1.0
IG = {"a": (3.0, 0.5), "b": (3.0, 0.5), "g": (3.0, 0.1), "d": (3.0, 0.05)}


def correr_m2(Y, N, seed=SEED + 1):
    rng = np.random.default_rng(seed)
    T = Y.shape[2]
    mu = rng.normal(M0, 0.2, K)
    a = rng.normal(0, 0.2, (K, L))
    b = rng.normal(0, 0.2, (K, F))
    g = rng.normal(0, 0.1, (K, L, F))
    d = rng.normal(0, 0.05, (K, T))
    s2 = {key: np.full(K, IG[key][1] / (IG[key][0] - 1)) for key in IG}

    def eta_de(mu, a, b, g, d):
        return mu[:, None, None, None] + a[:, :, None, None] + b[:, None, :, None] + g[..., None] + d[:, None, None, :]

    def ll_de(eta):
        return Y * eta - N * np.logaddexp(0.0, eta)

    escalas = {"mu": np.full(1, np.log(0.02)), "a": np.full(L, np.log(0.05)), "b": np.full(F, np.log(0.02)),
               "g": np.full((L, F), np.log(0.05)), "d": np.full(T, np.log(0.02)),
               "sh_mu_a": np.full(1, np.log(0.1)), "sh_mu_b": np.full(1, np.log(0.1)),
               "sh_mu_d": np.full(1, np.log(0.1)), "sh_a_g": np.full(L, np.log(0.1)),
               "sh_b_g": np.full(F, np.log(0.1))}
    acc = {k_: np.zeros_like(v_) for k_, v_ in escalas.items()}
    tasas = {k_: np.zeros_like(v_) for k_, v_ in escalas.items()}
    out = {k_: [] for k_ in ["mu", "sigma_a", "sigma_b", "sigma_g", "sigma_d"]}
    out["theta_new"] = np.empty((K, ITER, L * F), dtype=np.float32)
    out["a"] = np.empty((K, ITER, L), dtype=np.float32)
    out["b"] = np.empty((K, ITER, F), dtype=np.float32)
    out["d"] = np.empty((K, ITER, T), dtype=np.float32)

    eta = eta_de(mu, a, b, g, d)
    ll = ll_de(eta)
    for it in range(BURN + ITER):
        # --- mu
        e = np.exp(escalas["mu"][0]) * rng.standard_normal(K)
        eta_p = eta + e[:, None, None, None]
        llp = ll_de(eta_p)
        lr = (llp - ll).sum((1, 2, 3)) - ((mu + e - M0) ** 2 - (mu - M0) ** 2) / (2 * S0 ** 2)
        ok = np.log(rng.uniform(size=K)) < lr
        mu = np.where(ok, mu + e, mu)
        eta = np.where(ok[:, None, None, None], eta_p, eta)
        ll = np.where(ok[:, None, None, None], llp, ll)
        acc["mu"] += ok.mean()
        tasas["mu"] += ok.mean()
        # --- a_l (20 componentes; cada una solo afecta a sus filas)
        e = np.exp(escalas["a"]) * rng.standard_normal((K, L))
        eta_p = eta + e[:, :, None, None]
        llp = ll_de(eta_p)
        lr = (llp - ll).sum((2, 3)) - ((a + e) ** 2 - a ** 2) / (2 * s2["a"][:, None])
        ok = np.log(rng.uniform(size=(K, L))) < lr
        a = np.where(ok, a + e, a)
        m4 = ok[:, :, None, None]
        eta, ll = np.where(m4, eta_p, eta), np.where(m4, llp, ll)
        acc["a"] += ok.mean(0)
        tasas["a"] += ok.mean(0)
        # --- b_f
        e = np.exp(escalas["b"]) * rng.standard_normal((K, F))
        eta_p = eta + e[:, None, :, None]
        llp = ll_de(eta_p)
        lr = (llp - ll).sum((1, 3)) - ((b + e) ** 2 - b ** 2) / (2 * s2["b"][:, None])
        ok = np.log(rng.uniform(size=(K, F))) < lr
        b = np.where(ok, b + e, b)
        m4 = ok[:, None, :, None]
        eta, ll = np.where(m4, eta_p, eta), np.where(m4, llp, ll)
        acc["b"] += ok.mean(0)
        tasas["b"] += ok.mean(0)
        # --- g_lf (80 componentes)
        e = np.exp(escalas["g"]) * rng.standard_normal((K, L, F))
        eta_p = eta + e[..., None]
        llp = ll_de(eta_p)
        lr = (llp - ll).sum(3) - ((g + e) ** 2 - g ** 2) / (2 * s2["g"][:, None, None])
        ok = np.log(rng.uniform(size=(K, L, F))) < lr
        g = np.where(ok, g + e, g)
        m4 = ok[..., None]
        eta, ll = np.where(m4, eta_p, eta), np.where(m4, llp, ll)
        acc["g"] += ok.mean(0)
        tasas["g"] += ok.mean(0)
        # --- d_t
        e = np.exp(escalas["d"]) * rng.standard_normal((K, T))
        eta_p = eta + e[:, None, None, :]
        llp = ll_de(eta_p)
        lr = (llp - ll).sum((1, 2)) - ((d + e) ** 2 - d ** 2) / (2 * s2["d"][:, None])
        ok = np.log(rng.uniform(size=(K, T))) < lr
        d = np.where(ok, d + e, d)
        m4 = ok[:, None, None, :]
        eta, ll = np.where(m4, eta_p, eta), np.where(m4, llp, ll)
        acc["d"] += ok.mean(0)
        tasas["d"] += ok.mean(0)
        # --- Traslaciones (la verosimilitud no cambia; solo la razón de priors)
        #     mu + c, efectos - c  : reduce la correlación entre el intercepto y los efectos
        for clave, vec, s2k in (("sh_mu_a", a, "a"), ("sh_mu_b", b, "b"), ("sh_mu_d", d, "d")):
            c = np.exp(escalas[clave][0]) * rng.standard_normal(K)
            mu_p, vec_p = mu + c, vec - c[:, None]
            lr = (-((mu_p - M0) ** 2 - (mu - M0) ** 2) / (2 * S0 ** 2)
                  - ((vec_p ** 2).sum(1) - (vec ** 2).sum(1)) / (2 * s2[s2k]))
            ok = np.log(rng.uniform(size=K)) < lr
            mu = np.where(ok, mu_p, mu)
            vec[...] = np.where(ok[:, None], vec_p, vec)
            acc[clave] += ok.mean()
            tasas[clave] += ok.mean()
        #     a_l + c_l, g_l. - c_l   y   b_f + c_f, g_.f - c_f
        c = np.exp(escalas["sh_a_g"]) * rng.standard_normal((K, L))
        a_p, g_p = a + c, g - c[:, :, None]
        lr = (-((a_p ** 2 - a ** 2) / (2 * s2["a"][:, None]))
              - ((g_p ** 2 - g ** 2).sum(2) / (2 * s2["g"][:, None])))
        ok = np.log(rng.uniform(size=(K, L))) < lr
        a, g = np.where(ok, a_p, a), np.where(ok[:, :, None], g_p, g)
        acc["sh_a_g"] += ok.mean(0)
        tasas["sh_a_g"] += ok.mean(0)
        c = np.exp(escalas["sh_b_g"]) * rng.standard_normal((K, F))
        b_p, g_p = b + c, g - c[:, None, :]
        lr = (-((b_p ** 2 - b ** 2) / (2 * s2["b"][:, None]))
              - ((g_p ** 2 - g ** 2).sum(1) / (2 * s2["g"][:, None])))
        ok = np.log(rng.uniform(size=(K, F))) < lr
        b, g = np.where(ok, b_p, b), np.where(ok[:, None, :], g_p, g)
        acc["sh_b_g"] += ok.mean(0)
        tasas["sh_b_g"] += ok.mean(0)
        # --- Gibbs para las varianzas: s2 | x ~ IG(a0 + J/2, b0 + sum(x^2)/2)
        for key, x in (("a", a), ("b", b), ("g", g.reshape(K, -1)), ("d", d)):
            a0, b0 = IG[key]
            s2[key] = 1.0 / rng.gamma(a0 + x.shape[1] / 2, 1.0 / (b0 + (x ** 2).sum(1) / 2))
        # --- adaptación de escalas durante el calentamiento
        if 0 < it < BURN and it % 100 == 0:
            for k_ in escalas:
                tasa = acc[k_] / 100.0
                escalas[k_] = escalas[k_] + np.where(tasa > 0.44, 0.15, np.where(tasa < 0.25, -0.15, 0.0))
                acc[k_][...] = 0
        if it == BURN - 1:
            for k_ in tasas:
                tasas[k_][...] = 0
        if it >= BURN:
            j = it - BURN
            out["mu"].append(mu.copy())
            for key in "abgd":
                out["sigma_" + key].append(np.sqrt(s2[key]))
            # severidad esperada en un año nuevo: d_new ~ N(0, s2d)
            d_new = np.sqrt(s2["d"]) * rng.standard_normal(K)
            eta_new = mu[:, None, None] + a[:, :, None] + b[:, None, :] + g + d_new[:, None, None]
            out["theta_new"][:, j] = sp.expit(eta_new).reshape(K, -1)
            out["a"][:, j], out["b"][:, j], out["d"][:, j] = a, b, d
    for k_ in ["mu", "sigma_a", "sigma_b", "sigma_g", "sigma_d"]:
        out[k_] = np.stack(out[k_], axis=1)
    out["acept"] = {k_: float(np.mean(v_) / ITER) for k_, v_ in tasas.items()}
    return out

## Ajuste y diagnóstico

In [4]:
t0 = time.time()
m2 = correr_m2(Y3, N3)
print(f'Tiempo: {time.time() - t0:.1f} s')
print('Aceptación por bloque:', limpio(m2['acept']))
diag2 = resumen_az({'mu': m2['mu'], 'sigma_a': m2['sigma_a'], 'sigma_b': m2['sigma_b'], 'sigma_g': m2['sigma_g'],
                    'sigma_d': m2['sigma_d'], 'theta_new': m2['theta_new'].astype(float)})
diag2.to_csv(RES / 'diagnosticos_M2.csv')
print('R-hat máx:', diag2.r_hat.max(), '| ESS bulk mín:', diag2.ess_bulk.min(), '| ESS tail mín:', diag2.ess_tail.min())
diag2.loc[['mu', 'sigma_a', 'sigma_b', 'sigma_g', 'sigma_d']]

Tiempo: 11.9 s
Aceptación por bloque: {'mu': 0.38698, 'a': 0.36603, 'b': 0.37367, 'g': 0.36347, 'd': 0.36549, 'sh_mu_a': 0.32803, 'sh_mu_b': 0.38554, 'sh_mu_d': 0.36223, 'sh_a_g': 0.3563, 'sh_b_g': 0.34888}


R-hat máx: 1.0 | ESS bulk mín: 9204.0 | ESS tail mín: 21772.0


,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
mu,0.002,0.001,17868.0,24775.0,1.0
sigma_a,0.000,0.000,37827.0,57573.0,1.0
sigma_b,0.000,0.001,49307.0,59204.0,1.0
sigma_g,0.000,0.000,9204.0,21772.0,1.0
sigma_d,0.000,0.000,45333.0,55394.0,1.0


In [5]:
th2 = m2['theta_new'].reshape(-1, L * F).astype(float)
w = n_tr / n_tr.sum()
pmay2 = (th2 > (th2 * w).sum(1, keepdims=True)).mean(0)
bc = m2['b'].reshape(-1, F); bc = bc - bc.mean(1, keepdims=True)
dc = m2['d'].reshape(-1, 3)
np.savez_compressed(SALIDAS / 'm2_resumen.npz', mu=m2['mu'], sigma_a=m2['sigma_a'], sigma_b=m2['sigma_b'],
                    sigma_g=m2['sigma_g'], sigma_d=m2['sigma_d'], media=th2.mean(0), q05=np.quantile(th2, 0.05, 0),
                    q95=np.quantile(th2, 0.95, 0), p_top10=prob_top10(th2), p_mayor_ciudad=pmay2,
                    thin20=th2[::20].astype(np.float32))
R = {k: dict(media=m2[k].mean(), ci=np.quantile(m2[k], [0.05, 0.95]).tolist()) for k in ['mu', 'sigma_a', 'sigma_b', 'sigma_g', 'sigma_d']}
R['franja_OR'] = {FRANJAS[f]: dict(OR=np.exp(bc[:, f]).mean(), ci=np.exp(np.quantile(bc[:, f], [0.05, 0.95])).tolist()) for f in range(F)}
R['anio_d'] = {str(2015 + t): dc[:, t].mean() for t in range(3)}
R['acept'] = m2['acept']
R.update(rhat_max=diag2.r_hat.max(), ess_bulk_min=diag2.ess_bulk.min(), ess_tail_min=diag2.ess_tail.min())
print(limpio({k: R[k] for k in ['sigma_a', 'sigma_b', 'sigma_g', 'sigma_d', 'franja_OR']}))
guardar_json(R, '06_M2.json')

{'sigma_a': {'media': 0.42785, 'ci': [0.33305, 0.54963]}, 'sigma_b': {'media': 0.37429, 'ci': [0.25025, 0.55945]}, 'sigma_g': {'media': 0.15993, 'ci': [0.13098, 0.19352]}, 'sigma_d': {'media': 0.1405, 'ci': [0.09102, 0.21656]}, 'franja_OR': {'Madrugada (0-6)': {'OR': 1.09131, 'ci': [1.02419, 1.1605]}, 'Mañana (6-12)': {'OR': 0.952, 'ci': [0.89968, 1.00712]}, 'Tarde (12-18)': {'OR': 0.84405, 'ci': [0.79737, 0.89248]}, 'Noche (18-24)': {'OR': 1.14323, 'ci': [1.07903, 1.21006]}}}
Guardado: C:\Users\dddan\Downloads\Pensamiento - Bayesiana\4. Estadística Bayesiana\Proyecto\4. Entrega final\notebooks\salidas\resultados\06_M2.json
